In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance

import seaborn as sns

import matplotlib.pyplot as plt

import xgboost as xgb

import numpy as np

import skexplain



In [2]:
ENCODING_FILENAME = "./allpopulations_encoded.csv" # "./__encoding__solutions.csv"
df = pd.read_csv(ENCODING_FILENAME)
print("Dataframe shape:", df.shape)
print(df.columns)
df.head()

Dataframe shape: (2352, 118)
Index(['solID', 'sequence_len', 'num_new_nodes_created',
       'num_new_components_created', 'count_mcnn', 'count_moc', 'count_moncnn',
       'count_clone', 'bigram_mcnn_mcnn', 'bigram_mcnn_moc',
       ...
       'y__price', 'y__pricePerScenario__ProcessSale_job_class',
       'y__pricePerScenario__ReceivedOrderedProducts_job_class',
       'y__pricePerScenario__ShowDeliveryReports_job_class', 'y__reliability',
       'y__responseTimePerScenario__ProcessSale_job_class',
       'y__responseTimePerScenario__ReceivedOrderedProducts_job_class',
       'y__responseTimePerScenario__ShowDeliveryReports_job_class',
       'y__sysRespT', 'population_number'],
      dtype='object', length=118)


,solID,sequence_len,num_new_nodes_created,num_new_components_created,count_mcnn,count_moc,count_moncnn,count_clone,bigram_mcnn_mcnn,bigram_mcnn_moc,...,y__price,y__pricePerScenario__ProcessSale_job_class,y__pricePerScenario__ReceivedOrderedProducts_job_class,y__pricePerScenario__ShowDeliveryReports_job_class,y__reliability,y__responseTimePerScenario__ProcessSale_job_class,y__responseTimePerScenario__ReceivedOrderedProducts_job_class,y__responseTimePerScenario__ShowDeliveryReports_job_class,y__sysRespT,population_number
0,1,4,3,2,1,1,2,0,0,0,...,3.7753,2.354001,1.343544,0.077756,0.595982,654.297211,5.25794,1.39367,660.948821,1
1,6,4,3,0,1,1,0,2,0,0,...,4.2068,1.888328,0.759298,0.632773,-0.633613,879.086300,7.21529,1.94383,888.245420,1
2,4,4,4,1,1,0,1,2,0,0,...,4.3336,1.890800,1.002988,0.513412,-0.113465,2457.197030,5.34372,1.28623,2463.826980,1
3,3,4,3,2,0,1,2,1,0,0,...,3.8704,1.645012,0.899860,1.198728,-0.074655,694.365453,7.10604,1.80197,703.273463,1
4,11,4,4,4,0,0,4,0,0,0,...,4.6700,2.352360,1.318462,0.535977,-0.729991,654.157823,8.11287,1.29222,663.562913,1


In [3]:
x_cols = [col for col in df.columns if not col.startswith("y")]
y_cols = [col for col in df.columns if col.startswith("y")]

X = df[x_cols].drop(columns=["solID", "population_number"], axis=1)
y = df[["solID"] + y_cols].drop(columns=["solID"], axis=1)
print("objectives:", len(y_cols))

objectives: 17


In [4]:
# Identify columns where all values are zero
# cols_to_drop = X_morris.columns[(X_morris == 0).all()]

# Identify constant columns
constant_columns = [col for col in X.columns if X[col].nunique() == 1]
print(f"Constant columns identified: {constant_columns}")

# Drop these columns from the DataFrame
X = X.drop(columns=constant_columns, axis=1)

Constant columns identified: ['sequence_len', 'count_family_OTHER', 'count_family_c5', 'count_family_c6i', 'count_family_m5', 'count_family_r5', 'count_family_r6i', 'count_family_t3', 'count_mcnn_to_family_OTHER', 'count_mcnn_to_family_c5', 'count_mcnn_to_family_c6i', 'count_mcnn_to_family_m5', 'count_mcnn_to_family_r5', 'count_mcnn_to_family_r6i', 'count_mcnn_to_family_t3', 'count_moncnn_to_family_OTHER', 'count_moncnn_to_family_c5', 'count_moncnn_to_family_c6i', 'count_moncnn_to_family_m5', 'count_moncnn_to_family_r5', 'count_moncnn_to_family_r6i', 'count_moncnn_to_family_t3', 'count_clone_to_family_OTHER', 'count_clone_to_family_c5', 'count_clone_to_family_c6i', 'count_clone_to_family_m5', 'count_clone_to_family_r5', 'count_clone_to_family_r6i', 'count_clone_to_family_t2', 'count_clone_to_family_t3', 'count_size_12xlarge', 'count_size_16xlarge', 'count_size_24xlarge', 'count_size_32xlarge', 'count_size_4xlarge', 'count_size_8xlarge', 'count_size_OTHER', 'count_size_large', 'count_si

In [5]:
global_objectives = [
    'y__changes',
    # 'y__energy',
    # 'y__pas',
    # 'y__perfq',
    'y__power',
    'y__price',
    'y__reliability',
    # 'y__sysRespT'
 ]

In [8]:
XGBOOST_DEFAULT_PARAMS = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'n_estimators': 1000,
        'learning_rate': 0.1,
        'random_state': 42,
        # 'multi_strategy': 'one_output_per_tree'
    }

def train_xgboost(X, y, test_size=0.2, stratify=None, previous_model=None):

    if stratify is not None:
        print(f"\tapplying stratification on level!")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, stratify=stratify, random_state=42)

    model = xgb.XGBRegressor(**XGBOOST_DEFAULT_PARAMS)
    model.fit(X_train, y_train, xgb_model=previous_model)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse) 
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    print(f"\tTest metrics: R2={r2} -- RMSE={rmse} -- MAPE={(mape * 100):.2f}% ")
    return model, r2, rmse, mape

In [17]:
good_objectives = []
bad_objectives = []
estimators = []
for obj in global_objectives:
    print(f"Training model for objective: {obj}")
    previous_model = None
    model, r2, rmse, mape = train_xgboost(X, y[obj], test_size=0.3, stratify=df["population_number"])
    estimators.append((model.__class__.__name__, model))
    if r2 >= 0.7 and mape <= 0.5:
        good_objectives.append(obj)
    else:
        print(f"\tObjective {obj} is discarded")
        bad_objectives.append(obj)
print("Good objectives:", len(good_objectives), good_objectives)

Training model for objective: y__changes
	applying stratification on level!
	Test metrics: R2=0.8725209096218001 -- RMSE=0.7763617869117746 -- MAPE=3.87% 
Training model for objective: y__power
	applying stratification on level!
	Test metrics: R2=0.9323408997691867 -- RMSE=10.11776780979179 -- MAPE=1.76% 
Training model for objective: y__price
	applying stratification on level!
	Test metrics: R2=0.9929706108171703 -- RMSE=0.035572100019079136 -- MAPE=0.20% 
Training model for objective: y__reliability
	applying stratification on level!
	Test metrics: R2=0.9122469906416668 -- RMSE=0.1224145895169207 -- MAPE=26.59% 
Good objectives: 4 ['y__changes', 'y__power', 'y__price', 'y__reliability']


In [18]:
estimators

[('XGBRegressor',
  XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='rmse', feature_types=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.1, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=None, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=1000, n_jobs=None,
               num_parallel_tree=None, random_state=42, ...)),
 ('XGBRegressor',
  XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_roun

In [22]:
# estimators = skexplain.load_models()
last_objective = global_objectives[-1]
last_estimator = estimators[-1]
print(f"Creating ExplainToolkit for objective: {last_objective} using estimator: {last_estimator[0]}")
explainer = skexplain.ExplainToolkit(estimators=[last_estimator],X=X,y=y[last_objective])

Creating ExplainToolkit for objective: y__reliability using estimator: XGBRegressor


In [23]:
perm_results = explainer.permutation_importance(n_vars=5, evaluation_fn='auc')
explainer.plot_importance(data=perm_results)

ValueError: continuous format is not supported

In [ ]:
# Assumes the .permutation_importance has already been run.
important_vars = explainer.get_important_vars(results, multipass=True, nvars=7)

ale = explainer.ale(features=important_vars, n_bins=20)
explainer.plot_ale(ale)